<a href="https://colab.research.google.com/github/holehouse-lab/soursop/blob/master/demo_examples/ped_examples/colab/03_one_protein_many_methods.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This is the Google Colab version of this notebook. A copy with all its output is [in the SOURSOP repository](https://github.com/holehouse-lab/soursop/blob/master/demo_examples/ped_examples/03_one_protein_many_methods.ipynb).

# 3. One protein, many methods: alpha-synuclein

PED now holds ensembles of the same protein generated by many different methods. That makes it possible to ask how much the answer depends on the method, before a single simulation is run.

Here we take seven ensembles of full-length alpha-synuclein (140 residues, 100 conformers each), each from a different ensemble generator, and compare their global dimensions and polymer scaling with SOURSOP.

In [ ]:
# Colab does not come with SOURSOP, so install it first. This uses uv, which
# is much faster than pip (and is itself installed first if it is missing);
# --system installs into Colab's own Python rather than a virtual environment.
# The PED module is new in SOURSOP 2.0.8; if that version is not on PyPI yet,
# this installs the latest version from GitHub instead.
!command -v uv >/dev/null || pip install -q uv
!uv pip install --system -q "soursop>=2.0.8" 2>/dev/null || uv pip install --system -q "git+https://github.com/holehouse-lab/soursop.git"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from soursop import ped

# Figure style: Arial throughout, text kept editable in Illustrator, no grid
# lines, and the xmgrace colour order (black, red, blue, green, purple).
# Colab does not have Arial, so there figures fall back to Liberation Sans
# (metrically identical to Arial) or matplotlib's default, DejaVu Sans
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Liberation Sans", "DejaVu Sans"],
    "font.size": 10,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "axes.grid": False,
    "axes.prop_cycle": plt.cycler(color=["black", "red", "blue", "green", "purple"]),
})

# Downloaded ensembles are kept here and reused, so each is only fetched once
# per session (Colab deletes them when the session ends)
CACHE = "ped_cache"

## Choose the ensembles

These are all full-length alpha-synuclein ensemble predictions. The labels are the generator named in each entry's title.

In [ ]:
ENSEMBLES = {
    "AF-CALVADOS":    "PED20241e001",
    "IDPForge":       "PED17052e001",
    "bAIes":          "PED10839e001",
    "HyRes":          "PED08390e001",
    "COCOEMU":        "PED12536e001",
    "AFX-IDPCG":      "PED19330e001",
    "AFFLECTO-FRESA": "PED16526e001",
}

# titles from a single search (no per-entry requests needed)
titles = {e.entry_id: e.title for e in ped.search_entries(uniprot="P37840")}
for label, identifier in ENSEMBLES.items():
    entry_id, _ = ped.parse_identifier(identifier)
    print(f"{label:15s} {identifier}  {titles.get(entry_id, '')}")

## Load them

Each download is about 2-3 MB compressed; with `cache_dir` set, re-running the notebook reuses the files.

In [ ]:
proteins = {}
for label, identifier in ENSEMBLES.items():
    traj = ped.load_ensemble(identifier, cache_dir=CACHE)
    proteins[label] = traj.proteinTrajectoryList[0]
    p = proteins[label]
    print(f"{label:15s} {traj.n_frames} conformers, {p.n_residues} residues, {p.traj.n_atoms} atoms")

All seven have the same 140 residues, so their observables can be compared directly. (AFFLECTO-FRESA deposited heavy atoms only, hence the smaller atom count; the C-alpha-based observables below are unaffected.)

## Global dimensions

In [ ]:
results = {}
for label, p in proteins.items():
    results[label] = {
        "rg": p.get_radius_of_gyration(),
        "re": p.get_end_to_end_distance(mode="CA"),
        "asph": p.get_asphericity(verbose=False),
        "rh": p.get_hydrodynamic_radius(mode="kr"),
    }

print(f"{'method':15s}{'<Rg> (A)':>10s}{'<Re> (A)':>10s}{'<Rh> (A)':>10s}{'<asph>':>9s}")
for label, r in results.items():
    print(f"{label:15s}{r['rg'].mean():10.1f}{r['re'].mean():10.1f}{r['rh'].mean():10.1f}{r['asph'].mean():9.2f}")

means = [r["rg"].mean() for r in results.values()]
print(f"\nMean Rg ranges from {min(means):.1f} to {max(means):.1f} A across methods "
      f"({100 * (max(means) - min(means)) / np.mean(means):.0f}% of the average)")

The distributions show more than the means: some methods produce a narrow spread of sizes, others a much broader one.

In [ ]:
labels = list(results)
fig, ax = plt.subplots(figsize=(4.2, 3.0))
ax.boxplot(
    [results[k]["rg"] for k in labels],
    showfliers=False,
    medianprops={"color": "black"},
    boxprops={"color": "black"},
    whiskerprops={"color": "black"},
    capprops={"color": "black"},
)
for i, k in enumerate(labels, start=1):
    jitter = np.random.default_rng(i).uniform(-0.15, 0.15, len(results[k]["rg"]))
    ax.plot(i + jitter, results[k]["rg"], ".", color="black", ms=2, alpha=0.4)
ax.set_xticks(range(1, len(labels) + 1))
ax.set_xticklabels(labels, rotation=45, ha="right")
ax.set_ylabel("Radius of gyration (Å)")
plt.tight_layout()
plt.show()

## Polymer scaling

The internal-scaling profile, the root-mean-square distance between residues *i* and *j* as a function of |*i* - *j*|, shows where the differences come from: local chain stiffness at short separations, or long-range compaction at large ones. Up to five methods are shown for clarity.

In [ ]:
fig, ax = plt.subplots(figsize=(3.6, 2.9))
for label in labels[:5]:
    sep, rms = proteins[label].get_internal_scaling_RMS(mode="CA", verbose=False)
    ax.plot(sep[1:], rms[1:], lw=1, label=label)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Sequence separation |i - j| (residues)")
ax.set_ylabel("RMS Cα-Cα distance (Å)")
ax.legend(frameon=False, fontsize=7)
plt.tight_layout()
plt.show()

`get_scaling_exponent()` fits the apparent Flory exponent (and prefactor) to that profile, with bootstrap confidence intervals. For reference, a self-avoiding random coil has a scaling exponent of about 0.59, and an ideal (theta-solvent) chain has 0.5.

In [ ]:
nus = {}
for label, p in proteins.items():
    out = p.get_scaling_exponent(mode="CA", verbose=False)
    nus[label] = (out[0], out[2], out[3])      # nu, lower and upper CI

fig, ax = plt.subplots(figsize=(3.6, 2.9))
for i, label in enumerate(labels):
    nu, lo, hi = nus[label]
    ax.errorbar(nu, i, xerr=[[nu - lo], [hi - nu]], fmt="o", color="black", ms=4, capsize=2)
    print(f"{label:15s} nu = {nu:.3f}  (95% CI {lo:.3f}-{hi:.3f})")
ax.axvline(0.5, color="black", lw=0.8, ls=":")
ax.axvline(0.59, color="black", lw=0.8, ls="--")
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels)
ax.invert_yaxis()
ax.set_xlabel("Apparent scaling exponent ν")
plt.tight_layout()
plt.show()

The dotted and dashed lines mark the ideal-chain (0.5) and self-avoiding (0.59) limits.

Which ensemble is right cannot be decided from the ensembles alone. It needs experimental data, e.g. SAXS or smFRET, and the reweighting tools in `soursop.ssbme` and `soursop.sscoper`. What this comparison shows is how much the choice of method matters, and which observables are most sensitive to it.